# 🎓 Academic Advisor: Model Training (Google Colab)
Trains the classifier that predicts a student's **academic standing** (At Risk / Needs Support / On Track / Excellent) and exports `advisor_model.pkl` for the Flask web app.

**Steps:** install pinned libs → generate data → explore → compare models → evaluate → export `.pkl` → download.

In [ ]:
# 1. Pin the SAME library versions the web app uses (a .pkl must be loaded with the version that saved it)
!pip install -q scikit-learn==1.8.0 pandas==3.0.2 numpy==2.4.4 joblib==1.5.3
import sklearn; print("scikit-learn", sklearn.__version__)

## 2. Data
Real student records are private, so we simulate a realistic dataset. Swap in a real anonymised CSV later.

In [ ]:
'''Synthetic student data generator.

Real student records are private, so we simulate a realistic dataset.
Replace `generate_students` with a loader for real (anonymised) data later.
'''
import numpy as np
import pandas as pd

FEATURES = [
    "gpa",               # cumulative GPA on a 5.0 scale
    "attendance_pct",    # class attendance, 0-100
    "study_hours_week",  # self-study hours per week
    "failed_courses",    # number of failed/carryover courses
    "credit_load",       # credit units registered this semester
    "assignment_pct",    # assignment/CA completion rate, 0-100
    "part_time_job_hrs", # paid work hours per week
]
TARGET = "standing"
LABELS = ["At Risk", "Needs Support", "On Track", "Excellent"]


def generate_students(n: int = 3000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    attendance = np.clip(rng.normal(75, 15, n), 20, 100)
    study = np.clip(rng.normal(12, 6, n), 0, 40)
    assignment = np.clip(0.6 * attendance + rng.normal(25, 12, n), 0, 100)
    job = np.clip(rng.exponential(6, n), 0, 35)
    credit = rng.integers(12, 25, n)
    gpa = np.clip(
        1.2 + 0.018 * attendance + 0.045 * study + 0.012 * assignment
        - 0.02 * job + rng.normal(0, 0.35, n), 0.0, 5.0)
    failed = np.clip(np.round((3.2 - gpa) * 1.2 + rng.normal(0, 0.8, n)), 0, 8).astype(int)

    df = pd.DataFrame({
        "gpa": gpa.round(2), "attendance_pct": attendance.round(1),
        "study_hours_week": study.round(1), "failed_courses": failed,
        "credit_load": credit, "assignment_pct": assignment.round(1),
        "part_time_job_hrs": job.round(1),
    })
    score = (df.gpa * 1.0 + df.attendance_pct / 50 + df.assignment_pct / 60
             - df.failed_courses * 0.4 + rng.normal(0, 0.3, n))
    # quantile cut -> realistic class mix: ~12% At Risk, 23% Needs Support,
    # 40% On Track, 25% Excellent
    df[TARGET] = pd.qcut(score, q=[0, 0.12, 0.35, 0.75, 1.0],
                         labels=LABELS).astype(str)
    return df

In [ ]:
df = generate_students()
print(df.shape)
df.head()

## 3. Exploratory analysis

In [ ]:
import matplotlib.pyplot as plt
print(df[TARGET].value_counts(normalize=True).round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df[TARGET].value_counts().reindex(LABELS).plot.bar(ax=ax[0], title="Class balance", color="#0b5fff")
df.boxplot(column="gpa", by=TARGET, ax=ax[1]); ax[1].set_title("GPA by standing"); plt.suptitle("")
plt.tight_layout(); plt.show()
df[FEATURES].corr().round(2)

## 4. Train / test split and model comparison (5-fold cross-validation)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X_tr, X_te, y_tr, y_te = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, stratify=df[TARGET], random_state=42)

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}
for name, clf in candidates.items():
    pipe = Pipeline([("scale", StandardScaler()), ("clf", clf)])
    s = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring="f1_macro")
    print(f"{name:20s} macro-F1 = {s.mean():.3f} ± {s.std():.3f}")

## 5. Final model + evaluation on the held-out test set
Logistic Regression won cross-validation, and is also tiny (a few KB) and explainable, which is ideal for deployment.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
]).fit(X_tr, y_tr)

pred = model.predict(X_te)
print(classification_report(y_te, pred))
ConfusionMatrixDisplay.from_predictions(y_te, pred, labels=LABELS, xticks_rotation=30, cmap="Blues")
plt.show()

coef = pd.DataFrame(model.named_steps["clf"].coef_, index=model.classes_, columns=FEATURES)
imp = coef.abs().mean().sort_values()
imp.plot.barh(title="Feature influence (mean |coefficient|)", color="#0b5fff")
plt.tight_layout(); plt.show()

## 6. Export the `.pkl` and verify it reloads

In [ ]:
import joblib
joblib.dump(model, "advisor_model.pkl")

reloaded = joblib.load("advisor_model.pkl")
sample = pd.DataFrame([dict(gpa=1.4, attendance_pct=40, study_hours_week=2, failed_courses=5,
                            credit_load=24, assignment_pct=30, part_time_job_hrs=25)])
print("Sanity check →", reloaded.predict(sample)[0], reloaded.predict_proba(sample).round(2))

In [ ]:
from google.colab import files
files.download("advisor_model.pkl")   # then place it in the repo's models/ folder